# 01 BRONZE
## Raw source file → Bronze Delta table

Bronze goal:

```text
Preserve source structure
+
add ingestion metadata
+
store reliably as Delta
```

Input:

```text
orders.csv
126 rows
```

Output:

```text
demo_orders_bronze
126 rows
```


In [0]:
# ============================================================
# SETUP — RESOLVE CATALOG / SCHEMAS / OBJECT NAMES
# ============================================================
# WHAT:
#   Rebuild the same names used by the previous notebook.
#
# WHY:
#   Variables do not automatically persist between separate notebooks.
#
# EXPECTED:
#   All layer object names resolve consistently.

ctx = spark.sql("""
SELECT
    current_catalog() AS catalog,
    current_schema() AS schema
""").first()

CATALOG = ctx["catalog"]
CURRENT_SCHEMA = ctx["schema"]

if CATALOG == "hive_metastore":
    raise RuntimeError(
        "Switch to a Unity Catalog catalog first."
    )

def qi(value: str) -> str:
    return f"`{value.replace('`', '``')}`"

available = {
    r["databaseName"]
    for r in spark.sql(
        f"SHOW SCHEMAS IN {qi(CATALOG)}"
    ).collect()
}

BRONZE_SCHEMA = (
    "retail_bronze"
    if "retail_bronze" in available
    else CURRENT_SCHEMA
)

SILVER_SCHEMA = (
    "retail_silver"
    if "retail_silver" in available
    else CURRENT_SCHEMA
)

GOLD_SCHEMA = (
    "retail_gold"
    if "retail_gold" in available
    else CURRENT_SCHEMA
)

SOURCE_FILE = (
    f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/"
    f"demo_source_files/orders/orders.csv"
)

BRONZE_TABLE = (
    f"{qi(CATALOG)}.{qi(BRONZE_SCHEMA)}."
    f"{qi('demo_orders_bronze')}"
)

SILVER_TABLE = (
    f"{qi(CATALOG)}.{qi(SILVER_SCHEMA)}."
    f"{qi('demo_orders_silver')}"
)

GOLD_TABLE = (
    f"{qi(CATALOG)}.{qi(GOLD_SCHEMA)}."
    f"{qi('demo_order_status_gold')}"
)

print("Source :", SOURCE_FILE)
print("Bronze :", BRONZE_TABLE)
print("Silver :", SILVER_TABLE)
print("Gold   :", GOLD_TABLE)


In [0]:
# ============================================================
# CELL 2 — READ THE SOURCE WITH A SIMPLE RAW SCHEMA
# ============================================================
# WHAT:
#   Read orders.csv with all business columns as strings.
#
# WHY:
#   Bronze should stay close to the source and avoid too much business cleaning.
#
# INPUT GRAIN:
#   One CSV row = one order.
#
# OUTPUT GRAIN:
#   One Bronze DataFrame row = one raw order.
#
# EXPECTED:
#   126 rows.

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
)
from pyspark.sql import functions as F

ORDERS_RAW_SCHEMA = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_ts", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("shipping_city", StringType(), True),
    StructField("channel", StringType(), True),
])

bronze_df = (
    spark.read
    .option("header", True)
    .schema(ORDERS_RAW_SCHEMA)
    .csv(SOURCE_FILE)
    .withColumn(
        "source_file",
        F.input_file_name()
    )
    .withColumn(
        "ingestion_ts",
        F.current_timestamp()
    )
)

assert bronze_df.count() == 126

display(bronze_df)


In [0]:
# ============================================================
# CELL 3 — WRITE THE BRONZE DELTA TABLE
# ============================================================
# WHAT:
#   Persist the raw-ish DataFrame as a managed Delta table.
#
# WHY:
#   Delta gives us a reliable table layer for downstream Silver processing.
#
# EXPECTED:
#   demo_orders_bronze contains 126 rows.

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        BRONZE_TABLE
    )
)

bronze_count = spark.table(
    BRONZE_TABLE
).count()

print("Bronze rows:", bronze_count)

assert bronze_count == 126


In [0]:
# ============================================================
# CELL 4 — VERIFY BRONZE
# ============================================================
# WHAT:
#   Display a few Bronze rows and the table schema.
#
# WHY:
#   Students should confirm Bronze still looks close to the source.
#
# EXPECTED:
#   order_ts is still STRING and ingestion metadata is present.

spark.table(
    BRONZE_TABLE
).printSchema()

display(
    spark.table(
        BRONZE_TABLE
    ).limit(10)
)

print("\nBRONZE: PASSED")
print("Next notebook: 02_SILVER")
